# RAG 기본 구조 이해하기 — 8단계 파이프라인 완전 정복

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- RAG(Retrieval-Augmented Generation)의 개념과 왜 필요한지 설명하기
- 사전작업(Pre-processing) 4단계와 실행(Runtime) 4단계를 구분하기
- LangChain LCEL로 8단계를 하나의 체인으로 연결하기

## 사전 지식

- 6-1~6-4 섹션의 DocumentLoader, Chunking, Embedding, VectorStore 개념
- LangChain의 `|` 연산자(LCEL 파이프라인) 기본 문법

---

```mermaid
flowchart LR
    subgraph PRE[사전작업 Pre-processing]
        D[문서 로드]:::process --> S[청크 분할]:::process
        S --> E[임베딩 생성]:::process
        E --> V[(벡터 DB 저장)]:::storage
    end
    subgraph RT[실행 Runtime]
        Q[사용자 질문]:::input --> R[Retriever<br/>관련 문서 검색]:::process
        V --> R
        R --> P[Prompt<br/>문서+질문 조합]:::process
        P --> L[LLM<br/>답변 생성]:::process
        L --> A[최종 답변]:::output
    end

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
```

## RAG란?

**RAG(Retrieval-Augmented Generation, 검색 증강 생성)**는 외부 문서에서 관련 정보를 검색하여 LLM의 답변 품질을 향상시키는 기법이에요.

| 일반 LLM | RAG |
|---------|-----|
| 학습 데이터만 참조 | 외부 문서 실시간 참조 |
| 환각(Hallucination) 가능성 | 근거 기반 답변 |
| 지식 업데이트 어려움 | 문서만 교체하면 최신화 |

## RAG 파이프라인 개요

RAG는 크게 **사전작업(Pre-processing)**과 **실행 단계(Runtime)**로 나뉩니다.

### 1. 사전작업 (Pre-processing) - 1~4단계

문서를 Vector DB에 준비하는 단계입니다:

```
문서 → 로드 → 분할 → 임베딩 → Vector DB 저장
```

- **1단계 - 문서 로드(Document Load)**: 문서 파일을 읽어옵니다
- **2단계 - 분할(Text Split)**: 문서를 작은 청크(Chunk)로 나눕니다
- **3단계 - 임베딩(Embedding)**: 각 청크를 벡터로 변환합니다
- **4단계 - 벡터DB 저장**: 임베딩된 청크를 데이터베이스에 저장합니다

### 2. 실행 단계 (Runtime) - 5~8단계

사용자 질문에 답변하는 단계입니다:

```
사용자 질문 → 검색 → 프롬프트 생성 → LLM 호출 → 답변 생성
```

- **5단계 - 검색기(Retriever)**: 질문과 관련된 문서를 검색합니다
- **6단계 - 프롬프트**: 검색된 문서와 질문을 조합합니다
- **7단계 - LLM**: 언어 모델을 정의합니다
- **8단계 - 체인(Chain)**: 전체 파이프라인을 연결합니다

> 🎯 **강의 포인트**: RAG는 Ch06 전체의 클라이맥스입니다. 이전 6-1~6-6에서 배운 DocumentLoader, Chunking, Embedding, VectorStore, Retriever, Reranker가 모두 이 8단계 파이프라인으로 통합됩니다. 학생들에게 "지금까지 배운 것들이 퍼즐 조각이었고, 오늘 완성된 그림을 본다"고 강조하세요.

> 🔑 **핵심 개념**: Pre-processing(1~4단계)과 Runtime(5~8단계)의 구분이 RAG 아키텍처 이해의 핵심입니다. Pre-processing은 **한 번만** 실행되고, Runtime은 **사용자 질문마다** 실행됩니다.

## 환경 설정

In [1]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()

True

## 실습 문서

이번 실습에서는 디지털 정책 관련 문서를 사용합니다.

- 파일명: `sample-rag-brief.pdf`
- 위치: `../6-1_DocumentLoaders/data/` 디렉토리

**학습 목표**: PDF 문서를 기반으로 질문에 답변할 수 있는 RAG 시스템 구축

## RAG 기본 파이프라인 구현

이제 8단계 파이프라인을 순서대로 구현하겠습니다.

### 필수 라이브러리 import

In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

### 1단계: 문서 로드 (Load Documents)

`PyMuPDFLoader`를 사용하여 PDF 문서를 로드합니다.

**주요 특징**:
- PDF 파일을 페이지별로 읽어옵니다
- 각 페이지는 별도의 `Document` 객체로 저장됩니다
- 메타데이터(페이지 번호, 파일명 등)도 함께 저장됩니다

In [3]:
import os

# 데이터 파일 경로 확인
file_path = "./data/2026_gov.pdf"
if not os.path.exists(file_path):
    raise FileNotFoundError(f"파일을 찾을 수 없습니다: {file_path}\n현재 디렉토리: {os.getcwd()}")

In [4]:
# 단계 1: 문서 로드
loader = PyMuPDFLoader(file_path)
docs = loader.load()

print(f"문서의 페이지 수: {len(docs)}")
print(f"첫 번째 페이지 미리보기: {docs[0].page_content[:200]}...")

문서의 페이지 수: 501
첫 번째 페이지 미리보기: 발
간
등
록
번
호
11-1053000-100001-09
이렇게 
달라집니다
년부터
20
26...


특정 페이지의 전체 내용과 메타데이터를 확인해봅니다.

In [5]:
# 2번째 페이지 내용 출력
if len(docs) < 2:
    print("⚠️ 로드된 문서가 2개 미만이라 두 번째 페이지를 표시할 수 없습니다.")
else:
    print("=" * 60)
    print("[2번째 페이지 내용]")
    print("=" * 60)
    print(docs[1].page_content)
    print("\n" + "=" * 60)
    print("[메타데이터]")
    print("=" * 60)
    print(docs[1].metadata)


[2번째 페이지 내용]
이렇게 
달라집니다
년부터
20
26

[메타데이터]
{'producer': 'Adobe PDF Library 16.0.7', 'creator': 'Adobe InDesign 17.3 (Macintosh)', 'creationdate': '2026-01-05T22:01:47+09:00', 'source': './data/2026_gov.pdf', 'file_path': './data/2026_gov.pdf', 'total_pages': 501, 'format': 'PDF 1.6', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-01-05T22:01:47+09:00', 'trapped': '', 'modDate': "D:20260105220147+09'00'", 'creationDate': "D:20260105220147+09'00'", 'page': 1}


### 2단계: 문서 분할 (Split Documents)

`RecursiveCharacterTextSplitter`를 사용하여 문서를 작은 청크로 분할합니다.

**분할이 필요한 이유**:
- LLM의 컨텍스트 윈도우 크기 제한
- 검색 정확도 향상 (작은 단위가 더 정확)
- 관련 정보만 선택적으로 전달 가능

**주요 파라미터**:
- `chunk_size`: 각 청크의 최대 문자 수 (일반적으로 500~1500)
- `chunk_overlap`: 청크 간 겹치는 문자 수 (문맥 유지를 위해 10~20% 권장)

> ⚠️ **자주 하는 실수**: `chunk_size`와 `chunk_overlap` 값을 임의로 설정하는 경우가 많습니다. `chunk_overlap`은 `chunk_size`의 **10~20% 수준**이 적절합니다. 너무 크면 중복 데이터가 늘어 비효율적이고, 너무 작으면 문맥이 끊겨 검색 품질이 떨어집니다.

> 💡 **실무 팁**: 한국어 문서는 영어 대비 정보 밀도가 높으므로 `chunk_size=500~800`이 종종 더 좋은 결과를 줍니다. 도메인과 문서 유형에 따라 반드시 실험적으로 검증하세요.

In [6]:
# 단계 2: 문서 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # 각 청크의 최대 크기
    chunk_overlap=100  # 청크 간 겹치는 부분 (문맥 보존)
)
split_documents = text_splitter.split_documents(docs)

print(f"원본 문서 페이지 수: {len(docs)}")
print(f"분할된 청크 수: {len(split_documents)}")
print(f"\n첫 번째 청크 내용:\n{split_documents[0].page_content}")

원본 문서 페이지 수: 501
분할된 청크 수: 1034

첫 번째 청크 내용:
발
간
등
록
번
호
11-1053000-100001-09
이렇게 
달라집니다
년부터
20
26


### 3단계: 임베딩 (Embedding) 생성

`OpenAIEmbeddings`를 사용하여 텍스트를 벡터로 변환하는 임베딩 모델을 생성합니다.

**임베딩이란?**
- 텍스트를 고차원 벡터 공간의 점으로 표현하는 기술
- 의미가 유사한 텍스트는 벡터 공간에서 가까운 위치에 배치됨
- 이를 통해 의미 기반 검색(Semantic Search)이 가능해짐

In [7]:
# 단계 3: 임베딩 모델 생성
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large"  # OpenAI의 최신 임베딩 모델
)

print("임베딩 모델 생성 완료")
print(f"사용 모델: text-embedding-3-large")

임베딩 모델 생성 완료
사용 모델: text-embedding-3-large


### 4단계: 벡터 DB 생성 및 저장

`FAISS` 벡터스토어를 생성하고 분할된 문서를 임베딩하여 저장합니다.

**FAISS (Facebook AI Similarity Search)**:
- 메타(Meta)에서 개발한 고성능 벡터 검색 라이브러리
- 메모리 기반으로 빠른 검색 속도 제공
- 로컬 환경에서 간단히 사용 가능

In [9]:
# 단계 4: 벡터스토어 생성 및 문서 저장
vectorstore = FAISS.from_documents(
    documents=split_documents[:10],  # 분할된 문서
    embedding=embeddings  # 임베딩 모델
)

print(f"벡터스토어 생성 완료")
print(f"저장된 문서 청크 수: {len(split_documents)}")

벡터스토어 생성 완료
저장된 문서 청크 수: 1034


벡터스토어가 제대로 작동하는지 간단한 검색으로 테스트해봅니다.

In [10]:
# 벡터스토어 검색 테스트
test_query = "디지털 혁신"
search_results = vectorstore.similarity_search(test_query, k=2)

print(f"검색어: '{test_query}'")
print("=" * 60)
for i, doc in enumerate(search_results, 1):
    print(f"\n[검색 결과 {i}]")
    print(doc.page_content[:200] + "...")
    print(f"출처: 페이지 {doc.metadata.get('page', 'N/A')}")

검색어: '디지털 혁신'

[검색 결과 1]
018
생계형 창업중소기업 세액감면 적용 기준금액 상향
재정경제부
019
영상콘텐츠 세제지원 확대
재정경제부
020
부분복귀하는 해외진출기업에 대한 소득세·법인세·관세 감면 확대
재정경제부
021
위기지역 창업기업 등에 대한 세액감면 합리화
재정경제부
022
지방이전 기업 세제지원 제도 개선
재정경제부
023
장애인 표준사업장에 대한 세액감면 확대
재정경제...
출처: 페이지 4

[검색 결과 2]
이렇게 
달라집니다
년부터
20
26...
출처: 페이지 1


### 5단계: 검색기 (Retriever) 생성

벡터스토어를 `Retriever`로 변환합니다.

**Retriever의 역할**:
- 사용자 질문을 입력받아 관련 문서를 자동으로 검색
- 검색 알고리즘과 파라미터를 캡슐화
- 체인에서 바로 사용 가능한 형태로 제공

In [11]:
# 단계 5: 검색기 생성
retriever = vectorstore.as_retriever(
    search_type="similarity",  # 유사도 기반 검색
    search_kwargs={"k": 4}  # 상위 4개 문서 반환
)

print("검색기(Retriever) 생성 완료")
print(f"검색 방식: 유사도 기반 (similarity)")
print(f"반환 문서 수: 4개")

검색기(Retriever) 생성 완료
검색 방식: 유사도 기반 (similarity)
반환 문서 수: 4개


검색기에 실제 질문을 넣어 어떤 문서가 검색되는지 확인해봅니다.

In [12]:
# 검색기 테스트
test_question = "디지털 정부 혁신의 주요 목표는 무엇인가요?"
retrieved_docs = retriever.invoke(test_question)

print(f"질문: {test_question}")
print("=" * 60)
print(f"검색된 문서 수: {len(retrieved_docs)}\n")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"[문서 {i}]")
    print(doc.page_content[:150] + "...")
    print()

질문: 디지털 정부 혁신의 주요 목표는 무엇인가요?
검색된 문서 수: 4

[문서 1]
044
2025년 기준으로 소비자물가지수 개편
국가데이터처
045
근로·자녀장려금 압류금지 금액 상향
국세청
046
보세공장 제품의 과세방식 신청기한 수입신고 전까지 확대
관세청
047
개인통관고유부호 유효기간 도입
관세청
048
법규준수도 평가제도 통합으로 투명하고 ...

[문서 2]
교육부
061
취업 후 학자금대출 신청 대상 확대
교육부
062
방과후학교 지원 확대 및 온동네 초등돌봄 도입
교육부
063
아이돌봄서비스 정부 지원 확대
성평등가족부
064
한부모가족 아동양육비 등 복지지원 확대
성평등가족부
065
 
분야별 달라지는 주요 제도
ht...

[문서 3]
분야별 달라지는 주요 제도
01 금융·재정·조세
통합고용세액공제 공제액 구조 개편 및 사후관리 합리화
재정경제부
008
웹툰콘텐츠 제작비용 세액공제 신설
재정경제부
009
자녀 수에 따라 보육수당 비과세 한도 확대
재정경제부
010
초등 저학년 예체능 학원비 세제지원
...

[문서 4]
018
생계형 창업중소기업 세액감면 적용 기준금액 상향
재정경제부
019
영상콘텐츠 세제지원 확대
재정경제부
020
부분복귀하는 해외진출기업에 대한 소득세·법인세·관세 감면 확대
재정경제부
021
위기지역 창업기업 등에 대한 세액감면 합리화
재정경제부
022
지방이전 ...



### 6단계: 프롬프트 생성 (Create Prompt)

RAG를 위한 프롬프트 템플릿을 생성합니다.

**프롬프트 구성 요소**:
- `context`: 검색된 문서 내용이 자동으로 입력됨
- `question`: 사용자의 질문이 입력됨
- 시스템 지시사항: LLM에게 역할과 답변 방식을 명확히 전달

In [13]:
# 단계 6: 프롬프트 템플릿 정의
prompt = PromptTemplate.from_template(
    """당신은 문서 기반 질의응답을 수행하는 AI 어시스턴트입니다.
주어진 문맥(Context)을 바탕으로 질문(Question)에 답변해 주세요.

답변 시 주의사항:
- 문맥에서 찾을 수 있는 정보만 사용하세요
- 문맥에 없는 내용이라면 "주어진 문서에서 해당 정보를 찾을 수 없습니다"라고 답하세요
- 답변은 명확하고 간결하게 작성하세요
- 한국어로 답변하세요

#Context:
{context}

#Question:
{question}

#Answer:"""
)

print("프롬프트 템플릿 생성 완료")

프롬프트 템플릿 생성 완료


### 7단계: 언어 모델 (LLM) 생성

`ChatOpenAI` 모델을 생성합니다.

**모델 선택 가이드**:
- `gpt-4o-mini`: 빠르고 비용 효율적 (일반적인 QA에 적합)
- `gpt-4o`: 복잡한 추론이 필요한 경우
- `temperature=0`: 일관되고 결정적인 답변 생성 (RAG에서 권장)

> 🎯 **강의 포인트**: 아래 RAG 체인 생성 코드(`retriever | prompt | llm | parser`)가 이 챕터 **전체의 핵심**입니다. LCEL `|` 연산자 하나로 8단계 파이프라인이 연결됩니다. 이 4줄 코드를 화이트보드에 크게 써두고 각 구성 요소가 어떤 역할을 하는지 단계별로 설명하세요.

> 🔑 **핵심 개념**: `RunnablePassthrough()`는 입력(사용자 질문)을 그대로 `"question"` 키로 전달하는 역할입니다. `retriever`는 동일한 질문을 받아 자동으로 문서를 검색해 `"context"` 키로 전달합니다. 두 경로가 병렬로 실행된 뒤 `prompt`에서 합쳐집니다.

In [14]:
# 단계 7: 언어 모델 생성
llm = ChatOpenAI(
    model="gpt-4o-mini",  # 빠르고 경제적인 모델
    temperature=0  # 결정적인 출력 (일관성 확보)
)

print("언어 모델 생성 완료")
print(f"사용 모델: gpt-4o-mini")
print(f"Temperature: 0 (일관된 답변)")

언어 모델 생성 완료
사용 모델: gpt-4o-mini
Temperature: 0 (일관된 답변)


### 8단계: 체인 (Chain) 생성

이제 모든 구성 요소를 `|` 연산자로 연결하여 RAG 체인을 완성합니다.

**체인 흐름**:
```
1. 입력된 질문으로 문서 검색 (retriever)
2. 검색된 문서와 질문을 프롬프트에 삽입
3. LLM에 프롬프트 전달
4. LLM 응답을 문자열로 파싱
5. 최종 답변 반환
```

In [15]:
# 단계 8: RAG 체인 생성
rag_chain = (
    {
        "context": retriever,  # 질문으로 문서 검색
        "question": RunnablePassthrough()  # 질문을 그대로 전달
    }
    | prompt  # 프롬프트에 context와 question 삽입
    | llm  # LLM 호출
    | StrOutputParser()  # 출력을 문자열로 파싱
)

print("RAG 체인 생성 완료")
print("\n체인 구성:")
print("  1. Retriever → 문서 검색")
print("  2. Prompt → 템플릿 적용")
print("  3. LLM → 답변 생성")
print("  4. StrOutputParser → 문자열 변환")

RAG 체인 생성 완료

체인 구성:
  1. Retriever → 문서 검색
  2. Prompt → 템플릿 적용
  3. LLM → 답변 생성
  4. StrOutputParser → 문자열 변환


## RAG 체인 실행

이제 완성된 RAG 체인을 사용하여 문서에 대한 질문에 답변해봅니다.

> 🎯 **강의 포인트**: 질문 3은 **Hallucination 방지** 능력을 테스트합니다. 잘 구성된 RAG는 문서에 없는 내용에 대해 "모른다"고 답해야 합니다. 프롬프트의 "문맥에 없는 내용이라면..." 지시사항이 이를 제어합니다. 학생들에게 이 구문을 프롬프트에서 제거하면 어떤 일이 생기는지 직접 실험해보게 하세요.

In [16]:
# 질문 1: 문서의 주요 내용 파악
question1 = "이 문서의 주요 주제는 무엇인가요?"

print(f"질문: {question1}")
print("=" * 60)
answer1 = rag_chain.invoke(question1)
print(f"답변:\n{answer1}")

질문: 이 문서의 주요 주제는 무엇인가요?
답변:
주어진 문서에서 해당 정보를 찾을 수 없습니다.


In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# 1단계: 문서 로드
loader = PyMuPDFLoader("../6-1_DocumentLoaders/data/sample-rag-brief.pdf")
docs = loader.load()

# 2단계: 문서 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)
split_documents = text_splitter.split_documents(docs)

# 3단계: 임베딩 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 4단계: 벡터스토어 생성
vectorstore = FAISS.from_documents(
    documents=split_documents,
    embedding=embeddings
)

# 5단계: 검색기 생성
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

# 6단계: 프롬프트 생성
prompt = PromptTemplate.from_template(
    """당신은 문서 기반 질의응답을 수행하는 AI 어시스턴트입니다.
주어진 문맥(Context)을 바탕으로 질문(Question)에 답변해 주세요.

답변 시 주의사항:
- 문맥에서 찾을 수 있는 정보만 사용하세요
- 문맥에 없는 내용이라면 "주어진 문서에서 해당 정보를 찾을 수 없습니다"라고 답하세요
- 답변은 명확하고 간결하게 작성하세요
- 한국어로 답변하세요

#Context:
{context}

#Question:
{question}

#Answer:"""
)

# 7단계: LLM 생성
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 🎯 강의 포인트: 8단계 — LCEL | 연산자로 전체 파이프라인을 한 줄로 연결
# retriever → prompt → llm → parser 순서로 데이터가 흐릅니다
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}  # 💡 팁: dict로 두 경로 병렬 실행
    | prompt   # context + question → 완성된 프롬프트 문자열
    | llm      # 프롬프트 → AIMessage 응답
    | StrOutputParser()  # AIMessage → 순수 문자열
)

# 실행 예제
question = "이 문서의 주요 주제는 무엇인가요?"
answer = rag_chain.invoke(question)

print(f"질문: {question}")
print(f"답변: {answer}")

질문: 이 문서의 주요 주제는 무엇인가요?
답변: 주어진 문서에서 해당 정보를 찾을 수 없습니다.


In [18]:
# 질문 3: 문서에 없는 내용 질의 (모델의 답변 거절 능력 테스트)
question3 = "블록체인 기술의 미래 전망은 어떻게 되나요?"

print(f"질문: {question3}")
print("=" * 60)
answer3 = rag_chain.invoke(question3)
print(f"답변:\n{answer3}")

질문: 블록체인 기술의 미래 전망은 어떻게 되나요?
답변:
주어진 문서에서 해당 정보를 찾을 수 없습니다.


## 전체 코드 (통합 버전)

위에서 단계별로 구현한 RAG 파이프라인을 하나의 코드로 정리했습니다.

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# 1단계: 문서 로드
loader = PyMuPDFLoader(file_path)
docs = loader.load()

# 2단계: 문서 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)
split_documents = text_splitter.split_documents(docs)

# 3단계: 임베딩 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 4단계: 벡터스토어 생성
vectorstore = FAISS.from_documents(
    documents=split_documents[:10],
    embedding=embeddings
)

# 5단계: 검색기 생성
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

# 6단계: 프롬프트 생성
prompt = PromptTemplate.from_template(
    """당신은 문서 기반 질의응답을 수행하는 AI 어시스턴트입니다.
주어진 문맥(Context)을 바탕으로 질문(Question)에 답변해 주세요.

답변 시 주의사항:
- 문맥에서 찾을 수 있는 정보만 사용하세요
- 문맥에 없는 내용이라면 "주어진 문서에서 해당 정보를 찾을 수 없습니다"라고 답하세요
- 답변은 명확하고 간결하게 작성하세요
- 한국어로 답변하세요

#Context:
{context}

#Question:
{question}

#Answer:"""
)

# 7단계: LLM 생성
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 8단계: RAG 체인 생성
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 실행 예제
question = "이 문서의 주요 주제는 무엇인가요?"
answer = rag_chain.invoke(question)

print(f"질문: {question}")
print(f"답변: {answer}")

질문: 이 문서의 주요 주제는 무엇인가요?
답변: 이 문서의 주요 주제는 2026년부터 달라지는 주요 제도와 관련된 내용입니다. 특히 금융, 재정, 조세, 교육, 보육, 가족 등 다양한 분야에서의 제도 변화가 다루어지고 있습니다.


## 💡 핵심 정리

### RAG 파이프라인 8단계

1. **문서 로드**: PDF, 웹 등 다양한 형식의 문서를 읽어옴
2. **문서 분할**: 문서를 작은 청크로 나누어 검색 효율성 향상
3. **임베딩**: 텍스트를 벡터로 변환하여 의미 기반 검색 가능하게 함
4. **벡터 저장**: 임베딩된 청크를 벡터 DB에 저장
5. **검색기**: 질문과 관련된 문서를 자동으로 검색
6. **프롬프트**: 검색된 문서와 질문을 조합
7. **LLM**: 최종 답변 생성
8. **체인**: 전체 과정을 하나로 연결

### 주요 장점

- ✅ **정확성**: 외부 문서 기반으로 환각(Hallucination) 감소
- ✅ **최신성**: 문서만 업데이트하면 최신 정보 반영 가능
- ✅ **추적성**: 답변의 출처 문서 확인 가능
- ✅ **확장성**: 문서를 추가하여 지식 범위 확장 용이

### 개선 방향

다음 노트북에서는 RAG 성능을 향상시키는 고급 기법들을 다룹니다:
- Ensemble Retriever (여러 검색 방식 결합)
- Reranking (검색 결과 재정렬)
- Query Transformation (질문 개선)
- Contextual Compression (문맥 압축)

---

## 마무리 정리

### 8단계 RAG 파이프라인 요약

| 단계 | 구성 요소 | 핵심 역할 |
|------|-----------|-----------|
| 1. Load | `PyPDFLoader` | PDF에서 Document 객체 생성 |
| 2. Split | `RecursiveCharacterTextSplitter` | 청크 분할 (chunk_size/overlap) |
| 3. Embed | `OpenAIEmbeddings` | 텍스트 → 벡터 변환 |
| 4. Store | `FAISS` | 벡터 인덱스 저장 |
| 5. Retrieve | `.as_retriever()` | 쿼리와 유사한 문서 검색 |
| 6. Prompt | `ChatPromptTemplate` | 컨텍스트+질문 템플릿 결합 |
| 7. LLM | `ChatOpenAI` | 답변 생성 |
| 8. Chain | `|` (LCEL 파이프) | 전체 흐름 연결 |

### 핵심 파라미터 기준점

| 파라미터 | 기본값 | 조정 방향 |
|----------|--------|-----------|
| `chunk_size` | 1000 | 문서가 길면 늘리고, 정밀도가 필요하면 줄여요 |
| `chunk_overlap` | 200 | chunk_size의 10~20% 권장 |
| `k` (검색 문서 수) | 4 | 복잡한 질문엔 늘리고, 속도 중시엔 줄여요 |

### 다음 노트북 예고

**01-RAG-Web-Based** — PDF 대신 웹 페이지를 실시간으로 로드해 RAG를 구성하는 방법을 배워요. `WebBaseLoader`와 `BeautifulSoup`으로 Wikipedia 등 웹 문서를 바로 활용해요.